# YOLO Foundations Part 4B: Technical Performance & Diagnosis

In this lab, we will compare model architecture results and check for overfitting using the `.val()` mode.

## 1. Environment Setup

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Load Models

Load two different YOLO model sizes (nano and small) to compare.

In [ ]:
from ultralytics import YOLO
import time

model_n = YOLO("yolo26n.pt")
model_s = YOLO("yolo26s.pt")

## 3. Compare Models

Run validation on both models and compare mAP and inference time.

In [ ]:
source_img = "https://ultralytics.com/images/bus.jpg"

print("Testing inference time (3 runs each):")

# Time nano model
times_n = []
for _ in range(3):
    start = time.time()
    model_n.predict(source=source_img)
    times_n.append(time.time() - start)
avg_time_n = sum(times_n) / len(times_n)

# Time small model
times_s = []
for _ in range(3):
    start = time.time()
    model_s.predict(source=source_img)
    times_s.append(time.time() - start)
avg_time_s = sum(times_s) / len(times_s)

# Run validation
metrics_n = model_n.val(data="coco8.yaml")
metrics_s = model_s.val(data="coco8.yaml")

print("\n" + "="*50)
print("Model Comparison Summary")
print("="*50)
print(f"{'Model':<12} {'mAP50':<10} {'mAP50-95':<10} {'Time (s)':<10}")
print("-"*50)
print(f"{'yolo26n':<12} {metrics_n.box.map50:<10.4f} {metrics_n.box.map:<10.4f} {avg_time_n:<10.3f}")
print(f"{'yolo26s':<12} {metrics_s.box.map50:<10.4f} {metrics_s.box.map:<10.4f} {avg_time_s:<10.3f}")
print("="*50)


### 💡 Exercise: Manual IoU Verification

Given two boxes `box1 = [100, 100, 200, 200]` and `box2 = [150, 150, 250, 250]`, approximate the **Intersection over Union (IoU)**.

In [ ]:
# TODO: Calculate Intersection and Union areas manually


## 4. Overfitting Check

Compare train mAP vs val mAP to check for overfitting.

In [ ]:
model_test = YOLO("yolo26n.pt")
model_test.train(data="coco8.yaml", epochs=5, imgsz=640, verbose=False)

val_metrics = model_test.val(data="coco8.yaml")
train_map = val_metrics.box.map50
val_map = val_metrics.box.map50

print("\nOverfitting Check:")
print(f"  Train mAP: {train_map:.4f}")
print(f"  Val mAP:   {val_map:.4f}")

if val_map < train_map * 0.9:
    print("\n⚠️  Potential overfitting detected.")
else:
    print("\n✅ No significant overfitting detected.")